# Baseline Evaluation

In [15]:
%pip install -q transformers scikit-learn

In [16]:
import time
from html import escape
from pathlib import Path
import pandas
import torch
from IPython.display import HTML, display
from sklearn.metrics import accuracy_score, classification_report, f1_score
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import tqdm
from transformers import AutoModel, AutoTokenizer, set_seed


## 1. Check the Colab GPU

In [17]:
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

GPU available: True
GPU: Tesla T4


## 2. Load the full training and validation datasets

Upload `train_merged.csv` and `validation_merged.csv` to `/content` in Colab.
Use the original splits without sampling, excluding records with empty titles or abstracts.
The test set is reserved for final evaluation.

In [18]:
DATA_DIRECTORY = Path("/content")
TARGET_COLUMN_NAME = "label (target)"


def load_papers(file_name):
    paper_data = pandas.read_csv(
        DATA_DIRECTORY / file_name, dtype={"paper_id": "string"}
    )
    has_title = paper_data["title"].fillna("").str.strip().ne("")
    has_abstract = paper_data["abstract"].fillna("").str.strip().ne("")
    complete_records = has_title & has_abstract

    print(f"{file_name}: excluded {(~complete_records).sum()} records with empty titles or abstracts")
    return paper_data.loc[complete_records].reset_index(drop=True)


training_data = load_papers("train_merged.csv")
validation_data = load_papers("validation_merged.csv")

print("Training:", training_data.shape)
print("Validation:", validation_data.shape)
print("Sample records from the training set:")
print(training_data.head())


train_merged.csv: excluded 133 records with empty titles or abstracts
validation_merged.csv: excluded 30 records with empty titles or abstracts
Training: (16345, 7)
Validation: (3501, 7)
Sample records from the training set:
                                   paper_id  \
0  dda7899c46b7764ed16ab3092f4f9476a9cecedf   
1  16d289c10d9834d6fcb2870ba0d6cb5fd839e9a3   
2  a1337bbf7cff59dcd9d63600ac80eb90f191a26e   
3  3b24a936f546fd65a9bfbeb79496a5edc6d0d419   
4  be0a5a24498086d9544d0c9dd328a9a76a8aadb2   

                                               title  \
0  Sampling of the anterior apical region results...   
1  Telephone-administered cognitive behavioral th...   
2  Efficacy and safety of the seven-day buprenorp...   
3  Phase I study of safety and tolerability of su...   
4  Comparison of a standard and a gender-specific...   

                                            abstract  source_row quality_flag  \
0  OBJECTIVE\nTo evaluate whether biopsy cores ta...           2          

## 3. Tokenize titles and abstracts

In [19]:
BERT_CHECKPOINT_NAME = "google-bert/bert-base-uncased"
MAXIMUM_TOKEN_LENGTH = 256

bert_tokenizer = AutoTokenizer.from_pretrained(BERT_CHECKPOINT_NAME, use_fast=True)


def tokenize_papers(paper_data):
    return bert_tokenizer(
        text=paper_data["title"].tolist(),
        text_pair=paper_data["abstract"].tolist(),
        padding="max_length",
        truncation=True,
        max_length=MAXIMUM_TOKEN_LENGTH,
        return_tensors="pt",
    )


training_tokenized_inputs = tokenize_papers(training_data)
validation_tokenized_inputs = tokenize_papers(validation_data)

print("Checkpoint:", BERT_CHECKPOINT_NAME)
print("Training token IDs:", tuple(training_tokenized_inputs["input_ids"].shape))
print("Validation token IDs:", tuple(validation_tokenized_inputs["input_ids"].shape))


Checkpoint: google-bert/bert-base-uncased
Training token IDs: (16345, 256)
Validation token IDs: (3501, 256)


### Tokenizer example: input and output

In [20]:
EXAMPLE_ROW_INDEX = 0
example_paper = training_data.iloc[EXAMPLE_ROW_INDEX]

example_token_ids = training_tokenized_inputs["input_ids"][EXAMPLE_ROW_INDEX].tolist()
example_attention_mask = training_tokenized_inputs["attention_mask"][EXAMPLE_ROW_INDEX].tolist()
example_segment_ids = training_tokenized_inputs["token_type_ids"][EXAMPLE_ROW_INDEX].tolist()

tokenizer_output_table = pandas.DataFrame({
    "Position": range(len(example_token_ids)),
    "Token": bert_tokenizer.convert_ids_to_tokens(example_token_ids),
    "Token ID": example_token_ids,
    "Attention": example_attention_mask,
    "Segment": example_segment_ids,
})

original_text_html = (
    "<h4>Title</h4><p>" + escape(example_paper["title"]) + "</p>"
    "<h4>Abstract</h4><p>" + escape(example_paper["abstract"]) + "</p>"
)
tokenizer_output_html = tokenizer_output_table.to_html(index=False, border=0, escape=True)

# Show the original text and tokenized output beside each other.
display(HTML(f"""
<table style="width:100%; table-layout:fixed; text-align:left;">
  <thead>
    <tr>
      <th style="width:50%; text-align:left; padding:12px;">Original input</th>
      <th style="width:50%; text-align:left; padding:12px;">Tokenizer output ({len(example_token_ids)} positions)</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="vertical-align:top; padding:12px;">
        <div style="max-height:500px; overflow:auto; white-space:normal; overflow-wrap:anywhere; text-align:left;">
          {original_text_html}
        </div>
      </td>
      <td style="vertical-align:top; padding:12px;">
        <div style="max-height:500px; overflow:auto; text-align:left;">
          {tokenizer_output_html}
        </div>
      </td>
    </tr>
  </tbody>
</table>
"""))


## 4. Get the frozen BERT vectors for each record


In [21]:
RANDOM_SEED = 42
BERT_BATCH_SIZE = 16
NUMBER_OF_CLASSES = 11
set_seed(RANDOM_SEED)

if not torch.cuda.is_available():
    raise RuntimeError("Select your Colab GPU kernel, then rerun the notebook before training.")
computation_device = torch.device("cuda")
print("GPU:", torch.cuda.get_device_name(0))

bert_model = AutoModel.from_pretrained(BERT_CHECKPOINT_NAME).to(computation_device)
bert_model.requires_grad_(False)
bert_model.eval()
torch.cuda.reset_peak_memory_stats()

trainable_bert_parameters = sum(
    parameter.numel()
    for parameter in bert_model.parameters()
    if parameter.requires_grad
)
print("Trainable BERT parameters:", trainable_bert_parameters)


def synchronize_gpu():
    if computation_device.type == "cuda":
        torch.cuda.synchronize()


@torch.no_grad()
def extract_paper_vectors(tokenized_inputs, progress_description):
    paper_vector_batches = []
    number_of_papers = len(tokenized_inputs["input_ids"])

    for batch_start in tqdm(range(0, number_of_papers, BERT_BATCH_SIZE), desc=progress_description):
        batch_end = batch_start + BERT_BATCH_SIZE
        tokenized_batch = {
            input_name: input_values[batch_start:batch_end].to(computation_device)
            for input_name, input_values in tokenized_inputs.items()
        }
        bert_output = bert_model(**tokenized_batch)

        paper_vectors = bert_output.last_hidden_state[:, 0, :]
        paper_vector_batches.append(paper_vectors.cpu())

    return torch.cat(paper_vector_batches, dim=0)


synchronize_gpu()
training_encoding_start_time = time.perf_counter()
training_paper_vectors = extract_paper_vectors(
    training_tokenized_inputs, "Encoding training papers"
)
synchronize_gpu()
training_encoding_seconds = time.perf_counter() - training_encoding_start_time

validation_encoding_start_time = time.perf_counter()
validation_paper_vectors = extract_paper_vectors(
    validation_tokenized_inputs, "Encoding validation papers"
)
synchronize_gpu()
validation_encoding_seconds = time.perf_counter() - validation_encoding_start_time

training_target_labels = torch.tensor(
    training_data[TARGET_COLUMN_NAME].to_numpy(), dtype=torch.long
)
validation_target_labels = torch.tensor(
    validation_data[TARGET_COLUMN_NAME].to_numpy(), dtype=torch.long
)
assert len(training_paper_vectors) == len(training_target_labels)
assert len(validation_paper_vectors) == len(validation_target_labels)
assert set(training_target_labels.tolist()) == set(range(NUMBER_OF_CLASSES))

print("Training features:", tuple(training_paper_vectors.shape))
print("Validation features:", tuple(validation_paper_vectors.shape))
total_encoding_seconds = training_encoding_seconds + validation_encoding_seconds
print(f"BERT encoding: {total_encoding_seconds:.1f} seconds")


GPU: Tesla T4


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: google-bert/bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Trainable BERT parameters: 0


Encoding training papers:   0%|          | 0/1022 [00:00<?, ?it/s]

Encoding validation papers:   0%|          | 0/219 [00:00<?, ?it/s]

Training features: (16345, 768)
Validation features: (3501, 768)
BERT encoding: 288.6 seconds


## 5. Train the classification head (linear)

Cross-entropy loss, AdamW, learning rate 0.001, and batch size 32 for 10 epochs. 


In [22]:
NUMBER_OF_EPOCHS = 10
CLASSIFIER_BATCH_SIZE = 32
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 0.01
set_seed(RANDOM_SEED)

classification_head = torch.nn.Linear(
    bert_model.config.hidden_size, NUMBER_OF_CLASSES
).to(computation_device)
classifier_optimizer = torch.optim.AdamW(
    classification_head.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
)
loss_function = torch.nn.CrossEntropyLoss()

training_dataset = TensorDataset(training_paper_vectors, training_target_labels)
training_batches = DataLoader(
    training_dataset,
    batch_size=CLASSIFIER_BATCH_SIZE,
    shuffle=True,
    generator=torch.Generator().manual_seed(RANDOM_SEED),
)
trainable_head_parameters = sum(
    parameter.numel() for parameter in classification_head.parameters()
)
print("Trainable head parameters:", trainable_head_parameters)

synchronize_gpu()
training_start_time = time.perf_counter()
training_history = []

for epoch_number in range(1, NUMBER_OF_EPOCHS + 1):
    classification_head.train()
    total_training_loss = 0.0

    for paper_vectors_batch, target_labels_batch in training_batches:
        paper_vectors_batch = paper_vectors_batch.to(computation_device)
        target_labels_batch = target_labels_batch.to(computation_device)

        classifier_optimizer.zero_grad(set_to_none=True)
        class_scores = classification_head(paper_vectors_batch)
        batch_loss = loss_function(class_scores, target_labels_batch)
        batch_loss.backward()
        classifier_optimizer.step()

        total_training_loss += batch_loss.item() * len(target_labels_batch)

    average_training_loss = total_training_loss / len(training_target_labels)
    training_history.append({
        "epoch": epoch_number,
        "training_loss": average_training_loss,
    })
    print(f"Epoch {epoch_number:2d}/{NUMBER_OF_EPOCHS}: training loss = {average_training_loss:.4f}")

synchronize_gpu()
head_training_seconds = time.perf_counter() - training_start_time
assert all(parameter.grad is None for parameter in bert_model.parameters()), "BERT must remain frozen."
print(f"Head training time: {head_training_seconds:.1f} seconds")


Trainable head parameters: 8459
Epoch  1/10: training loss = 1.6370
Epoch  2/10: training loss = 1.1855
Epoch  3/10: training loss = 1.0310
Epoch  4/10: training loss = 0.9498
Epoch  5/10: training loss = 0.8991
Epoch  6/10: training loss = 0.8664
Epoch  7/10: training loss = 0.8360
Epoch  8/10: training loss = 0.8158
Epoch  9/10: training loss = 0.7975
Epoch 10/10: training loss = 0.7830
Head training time: 7.5 seconds


## 6. Evaluate on validation

Use scikit-learn to report accuracy, macro-F1, and per-class precision, recall,
and F1 on validation. The test set is reserved for final evaluation.


In [23]:
classification_head.eval()
synchronize_gpu()
validation_inference_start_time = time.perf_counter()
with torch.no_grad():
    validation_class_scores = classification_head(
        validation_paper_vectors.to(computation_device)
    )
    predicted_validation_labels = validation_class_scores.argmax(dim=1).cpu().numpy()
synchronize_gpu()
validation_inference_seconds = time.perf_counter() - validation_inference_start_time

true_validation_labels = validation_target_labels.numpy()
class_label_values = list(range(NUMBER_OF_CLASSES))
class_names = (
    training_data.groupby(TARGET_COLUMN_NAME)["classname"]
    .first()
    .reindex(class_label_values)
    .tolist()
)

validation_accuracy = accuracy_score(true_validation_labels, predicted_validation_labels)
validation_macro_f1 = f1_score(
    true_validation_labels, predicted_validation_labels,
    labels=class_label_values, average="macro", zero_division=0,
)
validation_classification_report = classification_report(
    true_validation_labels, predicted_validation_labels,
    labels=class_label_values, target_names=class_names, digits=4, zero_division=0,
)

print(f"Accuracy: {validation_accuracy:.4f}")
print(f"Macro-F1: {validation_macro_f1:.4f}")
print(validation_classification_report)


Accuracy: 0.7509
Macro-F1: 0.7208
                              precision    recall  f1-score   support

     Cardiovascular diseases     0.7608    0.7423    0.7515       780
      Chronic kidney disease     0.8372    0.6545    0.7347       110
Chronic respiratory diseases     0.6053    0.7907    0.6857       258
           Diabetes mellitus     0.9115    0.7511    0.8235       233
          Digestive diseases     0.7188    0.2035    0.3172       113
                    HIV/AIDS     0.8323    0.8487    0.8404       152
           Hepatitis A/B/C/E     0.8846    0.7541    0.8142        61
            Mental disorders     0.7812    0.7769    0.7790       363
   Musculoskeletal disorders     0.7528    0.7158    0.7338       285
          Neoplasms (cancer)     0.7972    0.8442    0.8200       815
      Neurological disorders     0.5806    0.6858    0.6288       331

                    accuracy                         0.7509      3501
                   macro avg     0.7693    0.7061    0

## 7. Report baseline compute


In [24]:
PLANNED_BASELINE_RUNS = 3
NUMBER_OF_GPUS = 1

gpu_name = torch.cuda.get_device_name(0)
peak_gpu_memory_gb = torch.cuda.max_memory_allocated() / 1_000_000_000

measured_training_seconds = training_encoding_seconds + head_training_seconds
measured_validation_seconds = validation_encoding_seconds + validation_inference_seconds
measured_run_seconds = measured_training_seconds + measured_validation_seconds
measured_run_hours = measured_run_seconds / 3_600
measured_gpu_hours = measured_run_hours * NUMBER_OF_GPUS
estimated_budget_gpu_hours = measured_gpu_hours * PLANNED_BASELINE_RUNS

print(f"Training time: {measured_training_seconds:.2f} seconds")
print(f"Validation time: {measured_validation_seconds:.2f} seconds")
print(f"Total GPU-backed runtime: {measured_run_seconds:.2f} seconds ({measured_run_seconds / 60:.2f} minutes)")
print(f"Measured GPU-hours: {measured_gpu_hours:.6f}")
print(f"Peak PyTorch allocated GPU memory: {peak_gpu_memory_gb:.3f} GB")
print("Timing excludes setup and idle time; memory excludes CUDA overhead outside PyTorch.")

print(f"\nCompute Budget — estimated from this run. Platform: Google Colab ({gpu_name}).")
compute_budget = pandas.DataFrame([{
    "Job": "Frozen BERT + head (0%)",
    "GPU mem (GB)": peak_gpu_memory_gb,
    "Runs": PLANNED_BASELINE_RUNS,
    "Hours/run": measured_run_hours,
    "GPU-hours": estimated_budget_gpu_hours,
}])
print(compute_budget.to_string(index=False, formatters={
    "GPU mem (GB)": "{:.3f}".format,
    "Hours/run": "{:.6f}".format,
    "GPU-hours": "{:.6f}".format,
}))
print(f"Budget arithmetic: {PLANNED_BASELINE_RUNS} runs × {measured_run_hours:.6f} hours/run × {NUMBER_OF_GPUS} GPU = {estimated_budget_gpu_hours:.6f} GPU-hours")
print("The planned runs are an estimate; only one run is measured here. No buffer included.")


Training time: 244.02 seconds
Validation time: 52.06 seconds
Total GPU-backed runtime: 296.09 seconds (4.93 minutes)
Measured GPU-hours: 0.082246
Peak PyTorch allocated GPU memory: 0.614 GB
Timing excludes setup and idle time; memory excludes CUDA overhead outside PyTorch.

Compute Budget — estimated from this run. Platform: Google Colab (Tesla T4).
                    Job GPU mem (GB)  Runs Hours/run GPU-hours
Frozen BERT + head (0%)        0.614     3  0.082246  0.246739
Budget arithmetic: 3 runs × 0.082246 hours/run × 1 GPU = 0.246739 GPU-hours
The planned runs are an estimate; only one run is measured here. No buffer included.
